# COMP 579 Assignment 3

Prepared by: Nicolas Smits

Presented to: Prof. Isabeau Premont Schwartz, Valliappan Chidambaram Adaikkappan

## Task 1 - Value Based Methods with Deep Neural Networks

In [1]:
import gym
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

if not hasattr(np, 'float_'):
    np.float_ = np.float64

if not hasattr(np, 'bool8'):
    np.bool8 = np.bool_

### 1.1 - Q-Learning and Expected SARSA with Deep NN Function Approximation

In [2]:
# Q-network Function Approximator
class QNetwork(nn.Module):
    def __init__(self, state_size, action_size, hidden_size=None):
        super(QNetwork, self).__init__()
        if hidden_size is None:
            hidden_size = [256, 256]
        layers = []
        last_dim = state_size # input layer init
        # populate hidden layers
        for hidden_dim in hidden_size:
            layers.append(nn.Linear(last_dim, hidden_dim))
            layers.append(nn.ReLU())
            last_dim = hidden_dim
        # output layer
        layers.append(nn.Linear(last_dim, action_size))
        self.model = nn.Sequential(*layers)
        self.apply(init_weights) # initialize weights (method in cell below)

    def forward(self, x):
        return self.model(x)
    
# Replay Buffer
class ReplayBuffer:
    def __init__(self, capacity):
        self.capacity = capacity
        self.buffer = []
        self.position = 0

    # store experience in replay buffer
    def push(self, state, action, reward, next_state, done):
        if len(self.buffer) < self.capacity: # buffer not full
            self.buffer.append(None) 
        self.buffer[self.position] = (state, action, reward, next_state, done) # store 
        self.position = (self.position + 1) % self.capacity # update position

    def sample(self, batch_size):
        batch = np.random.choice(self.buffer, batch_size, replace=False) # sample transitions randomly from the batch
        
        states, actions, rewards, next_states, dones = zip(*batch) # unzip the batch

        # return tuple of states, actions, rewards, next_states, and dones from the batch
        return (np.array(states), 
                np.array(actions), 
                np.array(rewards, dtype=np.float32), 
                np.array(next_states), 
                np.array(dones, dtype=np.uint8))
    
    def __len__(self):
        return len(self.buffer)

# Agent Superclass
class Agent:
    def __init__(self, env, alpha=1e-3, gamma=0.99, epsilon=0.1):
        self.env = env
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon

        # assume the obseration space is a vector
        state_space = env.observation_space.shape[0] # state space dimension
        action_space = env.action_space.n # action space dimension

        # for GPU acceleration
        if torch.backends.mps.is_available():
            self.device = torch.device("mps") # m1 macos gang wya
        elif torch.cuda.is_available():
            self.device = torch.device("cuda")
        else:
            self.device = torch.device("cpu")

        print(f"Using device: {self.device}")

        # Q-network
        self.q_network = QNetwork(state_space, action_space).to(self.device) # initialize Q-network
        self.optimizer = optim.Adam(self.q_network.parameters(), lr=alpha) # optimizer
        self.loss_fn = nn.MSELoss() # loss function
        # self.loss_fn = nn.SmoothL1Loss() # alternative
    
    # epsilon-greedy policy
    def select_action(self, state, greedy=False):
        if greedy:
            return self.q_network(torch.tensor(state, dtype=torch.float32).to(self.device)).argmax().item()
        else:
            if np.random.rand() < self.epsilon:
                return self.env.action_space.sample() # random action
            else:
                state_tensor = torch.FloatTensor(state).unsqueeze(0).to(self.device) # convert state to tensor
                with torch.no_grad(): # no gradient calculation
                        q_values = self.q_network(state_tensor) # greedy action
                return int(torch.argmax(q_values, dim=1).item())
            
    # update Q-network (general method, plug in target computation from agent)
    def update(self, s, a, r, s_, done):
        """
        Update Q-network.

        Args:
            s: current state
            a: current action
            r: reward
            s_: next state
            done: whether the episode is done
        Returns:
            loss: loss value
        """
        self.optimizer.zero_grad()

        state_tensor = torch.FloatTensor(s).unsqueeze(0).to(self.device)
        next_state_tensor = torch.FloatTensor(s_).unsqueeze(0).to(self.device)

        q_values = self.q_network(state_tensor)
        predicted = q_values[0, a]

        # The algorithm-specific target computation is done in compute_target.
        target = self.compute_target(r, next_state_tensor, done) ## ALGORITHM-DEPENDENT (subclasses)
        loss = self.loss_fn(predicted, target)
        loss.backward()
        self.optimizer.step()
        return loss.item()
    
    # update Q-network for a batch of transitions
    def update_batch(self, batch):
        """
        Update Q-network for a batch of transitions.

        Args:
            batch: tuple of (states, actions, rewards, next_states, dones)

        Returns:
            loss: loss value
        """
        states, actions, rewards, next_states, dones = batch

        states_tensor = torch.FloatTensor(states).to(self.device)
        actions_tensor = torch.LongTensor(actions).unsqueeze(1).to(self.device)
        rewards_tensor = torch.FloatTensor(rewards).to(self.device)
        next_states_tensor = torch.FloatTensor(next_states).to(self.device)
        dones_tensor = torch.FloatTensor(dones).to(self.device)

        q_values = self.q_network(states_tensor)
        # gather Q-values corresponding to taken actions.
        predicted = q_values.gather(1, actions_tensor).squeeze(1)
        
        target = self.compute_target_batch(rewards_tensor, next_states_tensor, dones_tensor) ## ALGORITHM-DEPENDENT (subclasses)
        loss = self.loss_fn(predicted, target)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        return loss.item()
    
    # methods are overridden by the subclass.
    def compute_target(self, reward, next_state_tensor, done):
        raise NotImplementedError("Override in subclass")

    def compute_target_batch(self, rewards_tensor, next_states_tensor, dones_tensor):
        raise NotImplementedError("Override in subclass")      

    
class QLearningAgent(Agent):
    def __init__(self, env, alpha=1e-3, gamma=0.99, epsilon=0.1):
        super(QLearningAgent, self).__init__(env, alpha, gamma, epsilon)

    # update Q-network via Q-learning
    # @overrides(Agent)
    def compute_target(self, reward, next_state_tensor, done):
        if done:
            return torch.tensor(reward, dtype=torch.float32, device=self.device)
        else:
            with torch.no_grad():
                next_q_values = self.q_network(next_state_tensor)
            # Q-learning uses the maximum over next-state Q-values.
            max_next_q = torch.max(next_q_values)
            target = torch.tensor(reward, dtype=torch.float32, device=self.device) + self.gamma * max_next_q # Q target
            return target
    
    # @overrides(Agent)
    def compute_target_batch(self, rewards_tensor, next_states_tensor, dones_tensor):
        with torch.no_grad():
            next_q_values = self.q_network(next_states_tensor)
            max_next_q_values, _ = next_q_values.max(dim=1)
        target = rewards_tensor + self.gamma * max_next_q_values * (1 - dones_tensor)
        return target

class ExpectedSarsaAgent(Agent):
    def __init__(self, env, alpha=0.001, gamma=0.99, epsilon=0.1):
        super().__init__(env, alpha, gamma, epsilon)

    # @overrides(Agent)
    def compute_target(self, reward, next_state_tensor, done):
        if done:
            return torch.tensor(reward, dtype=torch.float32, device=self.device)
        else:
            with torch.no_grad():
                # compute Q-values for the next state from the Q-network
                next_q_values = self.q_network(next_state_tensor)

            num_actions = next_q_values.size(1) # num actions

            # compute epsilon-greedy probabilities.
            best_action = torch.argmax(next_q_values, dim=1).item() # best action
            probs = torch.ones(num_actions, device=self.device) * (self.epsilon / num_actions) # epsilon-greedy probs
            probs[best_action] += (1.0 - self.epsilon) # update best action prob
            expected_q = torch.sum(next_q_values[0] * probs) 
            target = torch.tensor(reward, dtype=torch.float32, device=self.device) + self.gamma * expected_q
        
            return target
        
    # @override(Agent)
    def compute_target_batch(self, rewards_tensor, next_states_tensor, dones_tensor):
        with torch.no_grad():
            # compute Q-values for the next state from the Q-network
            next_q_values = self.q_network(next_states_tensor)
            num_actions = next_q_values.size(1) # num actions
            best_actions = next_q_values.argmax(dim=1) # best actions

            # build a probability tensor for the epsilon-greedy policy.
            probs = torch.ones_like(next_q_values) * (self.epsilon / num_actions)
            probs[range(probs.shape[0]), best_actions] += (1.0 - self.epsilon)
            expected_q = (next_q_values * probs).sum(dim=1)
        target = rewards_tensor + self.gamma * expected_q * (1 - dones_tensor)

        return target

# Training
def train_agent(agent, env, num_episodes=1000):
    """
    Train the agent in the given environment.

    Args:
        agent: Agent object
        env: OpenAI Gym environment
        num_episodes: number of episodes to train the agent
    Returns:
        None
    """
    agent.total_rewards = [] # store total rewards for each episode
    for episode in range(num_episodes):
        state, _ = env.reset()
        done = False
        total_reward = 0
        while not done:
            actions = env.action_space.n # get number of actions
            action = agent.select_action(state) # select action
            next_state, reward, terminated, truncated, info = env.step(action) # take action and observe next state and reward
            done = terminated or truncated
            total_reward += reward # update total reward
            loss = agent.update(state, action, reward, next_state, done) # update Q-network
            state = next_state
        agent.total_rewards.append(total_reward)
        print(f"Episode: {episode + 1}, Total Reward: {total_reward}")
    return agent.total_rewards


# Training with replay buffer and a batch update
def train_agent_with_replay(agent, env, num_episodes=1000, batch_size=32, start_training=1000, update_every=4):
    """
    Train the agent in the given environment.

    Args:
        agent: Agent object
        env: OpenAI Gym environment
        num_episodes: number of episodes to train the agent
        batch_size: batch size for updating the Q-network"
        start_training: start training after this number of transitions
        update_every: update the Q-network every this number of steps
    Returns:
        None
    """

    replay_buffer = ReplayBuffer() # initialize replay buffer
    total_steps = 0
        
    agent.total_rewards = [] # store total rewards for each episode
    for episode in range(num_episodes):
        state, _ = env.reset()
        done = False
        total_reward = 0
        total_loss = 0

        while not done:
            action = agent.select_action(state) # select action
            next_state, reward, terminated, truncated, info = env.step(action) # observe reward and next state
            done = terminated or truncated
            total_reward += reward
            
            # store transition in replay buffer
            replay_buffer.push(state, action, reward, next_state, done)
            state = next_state
            total_steps += 1

            # only update if we have enough transitions in the replay buffer
            if len(replay_buffer) > start_training and total_steps % update_every == 0:
                batch = replay_buffer.sample(batch_size) # sample from buffer
                loss = agent.update_batch(batch) # update based on batch
                total_loss += loss
            
        print(f"Episode: {episode + 1}, Total Reward: {total_reward}, Total Loss: {total_loss}")
    

### 1.2 - Model Configuration

In [3]:
# function to initialise Q-network weights
# called in the constructor of the QNetwork class (see above)
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.uniform_(m.weight, -0.001, 0.001)
        if m.bias is not None:
            nn.init.uniform_(m.bias, -0.001, 0.001)


### 1.3 - $\epsilon$-Greedy Policy for Different Initialisations

Acrobot v1 - Q-Learning

In [ ]:
# epsilon greedy policy params
epsilons = [0.05, 0.1, 0.2]
step_sizes = [0.125, 0.125, 0.0625]

# run params
num_trials = 10
num_episodes = 1000

### Q-learning Acrobot-v1 ###

# initialise environment
env = gym.make("Acrobot-v1")
# env = gym.make("Acrobot-v1", new_step_API=True, render_mode="human")

# initialise results
q_agent_acrobot_results = np.zeros((len(epsilons), len(step_sizes), num_trials, num_episodes))

# run experiments
for i, epsilon in enumerate(epsilons): 
    for j, step_size in enumerate(step_sizes):
        print(f"Running Experiment for Epsilon = {epsilon}, Step Size = {step_size}")
        for trial in range(num_trials):
            # initialise agent
            agent = QLearningAgent(env, alpha=step_size, gamma=0.99, epsilon=epsilon)
            # train the agent
            rewards = train_agent(agent, env, num_episodes)
            # record results
            q_agent_acrobot_results[i, j, trial, :] = np.array(rewards)
            print(f"Completed Trial {trial+1} for Epsilon = {epsilon}, Step Size = {step_size}")

# save results
np.save("q_agent_acrobot_results.npy", q_agent_acrobot_results)


Running Experiment for Epsilon = 0.1, Step Size = 0.125
Using device: mps
Episode: 1, Total Reward: -500.0
Episode: 2, Total Reward: -500.0
Episode: 3, Total Reward: -500.0
Episode: 4, Total Reward: -500.0
Episode: 5, Total Reward: -500.0
Episode: 6, Total Reward: -500.0
Episode: 7, Total Reward: -500.0
Episode: 8, Total Reward: -500.0
Episode: 9, Total Reward: -500.0
Episode: 10, Total Reward: -500.0
Episode: 11, Total Reward: -500.0
Episode: 12, Total Reward: -500.0
Episode: 13, Total Reward: -500.0
Episode: 14, Total Reward: -500.0
Episode: 15, Total Reward: -500.0
Episode: 16, Total Reward: -500.0
Episode: 17, Total Reward: -500.0
Episode: 18, Total Reward: -500.0
Episode: 19, Total Reward: -500.0
Episode: 20, Total Reward: -500.0
Episode: 21, Total Reward: -500.0
Episode: 22, Total Reward: -500.0
Episode: 23, Total Reward: -500.0
Episode: 24, Total Reward: -500.0
Episode: 25, Total Reward: -500.0
Episode: 26, Total Reward: -500.0
Episode: 27, Total Reward: -500.0
Episode: 28, Tota

KeyboardInterrupt: 

Acrobot v1 - Expected SARSA

In [ ]:
### Expected Sarsa Acrobot-v1 ###
num_trials = 10

# initialise environment
env = gym.make("Acrobot-v1")

# initialise results
expected_sarsa_agent_acrobot_results = np.zeros((len(epsilons), len(step_sizes), num_trials, num_episodes))

# run experiments
for i, epsilon in enumerate(epsilons):
    for j, step_size in enumerate(step_sizes):
        print(f"Running Experiment for Epsilon = {epsilon}, Step Size = {step_size}")
        for trial in range(num_trials):
            # initialise agent
            agent = ExpectedSarsaAgent(env, epsilon=epsilon)
            # train the agent
            rewards = train_agent(agent, env, num_episodes)
            # record results
            expected_sarsa_agent_acrobot_results[i, j, trial, :] = np.array(rewards)
            print(f"Completed Trial {trial+1} for Epsilon = {epsilon}, Step Size = {step_size}")

# save results
np.save("expected_sarsa_agent_acrobot_results.npy", expected_sarsa_agent_acrobot_results)

ALE/Assault-ram-v5 - Q-Learning

In [ ]:
# initialise environment
env = gym.make("ALE/Assault-ram-v5")

# initialise results
q_agent_ale_results = np.zeros((len(epsilons), len(step_sizes), num_trials, num_episodes))

# run experiments
for i, epsilon in enumerate(epsilons):
    for j, step_size in enumerate(step_sizes):
        print(f"Running Experiment for Epsilon = {epsilon}, Step Size = {step_size}")
        for trial in range(num_trials):
            # initialise agent
            agent = QLearningAgent(env, epsilon=epsilon)
            # train the agent
            rewards = train_agent(agent, env, num_episodes)
            # record results
            q_agent_ale_results[i, j, trial, :] = np.array(rewards)
            print(f"Completed Trial {trial+1} for Epsilon = {epsilon}, Step Size = {step_size}")

# save results
np.save("q_agent_ale_results.npy", q_agent_ale_results)


ALE/Assault-ram-v5 - Expected SARSA

In [ ]:
# initialise environment
env = gym.make("ALE/Assault-ram-v5")

# initialise results
expected_sarsa_agent_ale_results = np.zeros((len(epsilons), len(step_sizes), num_trials, num_episodes))

# run experiments
for i, epsilon in enumerate(epsilons):
    for j, step_size in enumerate(step_sizes):
        print(f"Running Experiment for Epsilon = {epsilon}, Step Size = {step_size}")
        for trial in range(num_trials):
            # initialise agent
            agent = ExpectedSarsaAgent(env, epsilon=epsilon)
            # train the agent
            rewards = train_agent(agent, env, num_episodes)
            # record results
            expected_sarsa_agent_ale_results[i, j, trial, :] = np.array(rewards)
            print(f"Completed Trial {trial+1} for Epsilon = {epsilon}, Step Size = {step_size}")

# save results
np.save("expected_sarsa_agent_ale_results.npy", expected_sarsa_agent_ale_results)


### 1.4 - Repeat Using Replay Buffer

Acrobot-v1 - Q-Learning with Buffer

Acrobot-v1 - Expected SARSA with Buffer

ALE/Assault-ram-v5 - Q-Learning with Buffer

ALE/Assault-ram-v5 - Expected SARSA with Buffer